# Preprocessing: Dataset A (team procedures)

This notebook applies the preprocessing the team agreed on in `preprocessing-procedures.md`. The result is **Dataset A**, the shared cleaned dataset everyone uses so modeling results can be compared. `02_preprocessing.ipynb` is my earlier, individual version. Section 6 lists where the two differ.

The code is in `idx_exchange_data_science.dataset_a`. As the procedures doc requires, the work is split in two:

1. **Fixed cleaning rules** don't learn anything from the data, so they run once on all months. Their output is the cleaned CSV.
2. **Training-derived steps** are fit on the training window only and then applied to the test month: ClosePrice outlier limits, imputation, missing flags, and target encoding.

**Input:** `data/intermediate/CRMLSSoldFiltered.csv`  **Output:** `data/processed/CRMLSSoldDatasetA.csv`

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from idx_exchange_data_science import dataset_a as da

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.3f}".format)

RAW_PATH = Path("../data/intermediate/CRMLSSoldFiltered.csv")
OUT_PATH = Path("../data/processed/CRMLSSoldDatasetA.csv")
TEST_MONTH = "2026-04"

In [2]:
raw = da.load_raw(RAW_PATH)
raw.shape

(309735, 84)

## 1. Fixed cleaning rules
| Rule (procedures doc) | Step |
|---|---|
| Keep `StateOrProvince == "CA"`, Residential / SingleFamilyResidence | `keep_scope` |
| Remove missing or non-positive `ClosePrice` | `drop_invalid_prices` |
| Remove `CloseDate` earlier than `ListingContractDate` | `drop_closed_before_listed` |
| Same valid address + same `CloseDate` is a duplicate, keep the first. Drop missing or invalid addresses | `drop_duplicate_sales` |
| Remove zero or negative `LivingArea` | `drop_nonpositive_living_area` |
| Value fixes (ranges, ZIP, lot size, `PropertyAge`), see section 2 | `fix_fields` |
| Agreed feature list, minus any column over 80% missing | `select_columns` |

Rows left after each step:

In [3]:
df = raw
counts = [("loaded", len(df))]
for step in da.CLEANING_STEPS:
    df = step(df)
    counts.append((step.__name__, len(df)))
pd.DataFrame(counts, columns=["step", "rows"]).assign(removed=lambda t: -t["rows"].diff().fillna(0).astype(int))

,step,rows,removed
0,loaded,309735,0
1,keep_scope,309713,22
2,parse_dates,309713,0
3,drop_invalid_prices,309710,3
4,drop_closed_before_listed,309666,44
5,drop_duplicate_sales,309145,521
6,drop_nonpositive_living_area,309019,126
7,fix_fields,309019,0
8,select_columns,309019,0


### Duplicates
Addresses are normalized first (lowercase, trimmed, repeated spaces collapsed), as Emily suggested. An address counts as **invalid** if it has no street number, e.g. `" Calle De La Estrella"`. Those can't be matched reliably.

Soonbee suggested flagging pairs that share an address and date but have different prices. They are counted below. The agreed rule still keeps the first one.

In [4]:
scoped = da.drop_closed_before_listed(da.drop_invalid_prices(da.parse_dates(da.keep_scope(raw))))
address = da.normalize_address(scoped["UnparsedAddress"])
print("missing address:", scoped["UnparsedAddress"].isna().sum())
print("no street number:", (scoped["UnparsedAddress"].notna() & address.isna()).sum())

sales = scoped.assign(address=address).dropna(subset=["address"])
groups = sales[sales.duplicated(["address", "CloseDate"], keep=False)].groupby(["address", "CloseDate"])
print("duplicate groups:", groups.ngroups, "| with different prices:", (groups["ClosePrice"].nunique() > 1).sum())
print("ListingKey still duplicated after the rule:", df[da.ID_COL].duplicated().sum())

missing address: 294
no street number: 43
duplicate groups: 182 | with different prices: 7
ListingKey still duplicated after the rule: 233


## 2. Value fixes
`fix_fields` only changes values, it doesn't drop rows:

- **Ranges:** `BedroomsTotal` and `BathroomsTotalInteger` 1 to 20, `GarageSpaces` and `ParkingTotal` 0 to 20, latitude 32.5 to 42.1, longitude -124.5 to -114.0. Values outside become missing and are imputed later.
- **Lot size:** zero or negative `LotSizeSquareFeet` becomes missing, then is recovered from `LotSizeAcres × 43,560` where possible. `LotSizeAcres` is dropped after that.
- **PostalCode:** first five digits as a string (`94513-9999` becomes `94513`). Codes that don't start with five digits become missing.
- **PropertyAge** = `CloseDate.year - YearBuilt`. Replaces `YearBuilt`.
- **Y/N fields** become 1/0, with missing kept as missing.
- **Stories** and **Flooring** are not used (doc: "Remove this feature", "Not for Dataset A").

In [5]:
before = da.drop_nonpositive_living_area(da.drop_duplicate_sales(scoped))
after = da.fix_fields(before)
nulled = {col: int(after[col].isna().sum() - before[col].isna().sum()) for col in da.VALID_RANGES}
lot_missing_before = (before["LotSizeSquareFeet"].isna() | (before["LotSizeSquareFeet"] <= 0)).sum()
print("values set to missing by range checks:", nulled)
print("lot size missing or <= 0:", lot_missing_before, "| still missing after using acres:", after["LotSizeSquareFeet"].isna().sum())
print("PostalCode set to missing:", after["PostalCode"].isna().sum() - before["PostalCode"].isna().sum())

values set to missing by range checks: {'BedroomsTotal': 141, 'BathroomsTotalInteger': 104, 'GarageSpaces': 48, 'ParkingTotal': 672, 'Latitude': 36, 'Longitude': 54}
lot size missing or <= 0: 5445 | still missing after using acres: 5425
PostalCode set to missing: 0


### High-missingness columns
The rule is to drop columns over 80% missing. These are the raw columns it would remove. None of them are on the agreed feature list anyway. `select_columns` checks the agreed features against the same threshold:

In [6]:
missing_share = before.isna().mean()
print(missing_share[missing_share > da.MAX_MISSING_SHARE].sort_values(ascending=False).round(3).to_string())
print("\nhighest missing share among agreed features:", round(df[da.FEATURES].isna().mean().max(), 3))

ElementarySchoolDistrict       1.000
TaxYear                        1.000
FireplacesTotal                1.000
AboveGradeFinishedArea         1.000
TaxAnnualAmount                1.000
MiddleOrJuniorSchoolDistrict   1.000
CoveredSpaces                  1.000
BusinessType                   1.000
WaterfrontYN                   1.000
BelowGradeFinishedArea         0.994
BasementYN                     0.976
BuilderName                    0.953
LotSizeDimensions              0.939
BuildingAreaTotal              0.937
CoBuyerAgentFirstName          0.910
OriginatingSystemName          0.893
OriginatingSystemSubName       0.893
BuyerAgencyCompensation        0.864
BuyerAgencyCompensationType    0.864
ElementarySchool               0.863
MiddleOrJuniorSchool           0.862
HighSchool                     0.821

highest missing share among agreed features: 0.116


## 3. Save Dataset A
Missing values are left in. Imputation and encoding have to be fit on whichever training window a model uses.

In [7]:
df.to_csv(OUT_PATH, index=False)
reloaded = da.load_clean(OUT_PATH)
pd.testing.assert_frame_equal(reloaded, df, check_dtype=False)
print(OUT_PATH, df.shape)
df.isna().mean().mul(100).round(2).rename("% missing").to_frame().T

../data/processed/CRMLSSoldDatasetA.csv (309019, 20)


,ListingKey,CloseDate,ClosePrice,LivingArea,PropertyAge,LotSizeSquareFeet,BedroomsTotal,BathroomsTotalInteger,GarageSpaces,ParkingTotal,AttachedGarageYN,FireplaceYN,NewConstructionYN,PoolPrivateYN,ViewYN,Latitude,Longitude,City,CountyOrParish,PostalCode
% missing,0.000,0.000,0.000,0.050,0.060,1.760,0.050,0.050,3.710,0.330,11.600,0.070,7.260,9.600,8.890,0.860,0.870,0.080,0.000,0.000


## 4. Training-derived steps
### ClosePrice outliers
The 0.5th and 99.5th percentiles of `ClosePrice` come from the **training window**. Both training and test rows outside them are dropped. `da.split` does the time split and then this trim. With a 6-month window before the test month:

In [8]:
train_raw, test_raw = da.time_split(df, TEST_MONTH, train_months=6)
low, high = da.price_limits(train_raw)
train, test = da.split(df, TEST_MONTH, train_months=6)
print(f"limits: ${low:,.0f} to ${high:,.0f}")
print(f"train {len(train_raw):,} -> {len(train):,} rows, test {len(test_raw):,} -> {len(test):,} rows")

limits: $180,000 to $9,018,333
train 60,119 -> 59,523 rows, test 12,445 -> 12,329 rows


### Imputation and encoding
`DatasetAPreprocessor` is an sklearn transformer, so it can be the first step of any model pipeline:

| Feature | Missing values |
|---|---|
| `LivingArea`, `PropertyAge`, `LotSizeSquareFeet` | training median |
| `BedroomsTotal`, `BathroomsTotalInteger`, `GarageSpaces`, `ParkingTotal` | training mode |
| 5 Y/N fields | `<field>_missing` indicator, then training mode |
| `Latitude`, `Longitude` | training median of the same ZIP, then county, then overall |
| `City`, `CountyOrParish`, `PostalCode` | `"Unknown"`, then target encoding, cross-fitted over 5 folds |

`log_features=True` log-transforms `LivingArea` and `LotSizeSquareFeet`. This is the experiment for linear models. Scaling isn't in the preprocessor. Models that need it add a `StandardScaler` after it (see `modeling.build_model`).

In [9]:
pre = da.DatasetAPreprocessor()
X_train = pre.fit_transform(train[da.FEATURES], train[da.TARGET])
X_test = pre.transform(test[da.FEATURES])
print(len(pre.get_feature_names_out()), "features")
print("NaNs left:", int(X_train.isna().sum().sum()), "/", int(X_test.isna().sum().sum()))
X_train.head()

22 features
NaNs left: 0 / 0


,LivingArea,PropertyAge,LotSizeSquareFeet,BedroomsTotal,BathroomsTotalInteger,GarageSpaces,ParkingTotal,AttachedGarageYN,FireplaceYN,NewConstructionYN,PoolPrivateYN,ViewYN,Latitude,Longitude,AttachedGarageYN_missing,FireplaceYN_missing,NewConstructionYN_missing,PoolPrivateYN_missing,ViewYN_missing,City_te,CountyOrParish_te,PostalCode_te
0,"2,014.000",50.000,"7,241.000",3.000,3.000,2.000,7.000,1.000,0.000,0.000,0.000,0.000,32.819,-116.776,0.000,0.000,0.000,0.000,0.000,"1,082,604.127","1,342,310.376","1,082,604.127"
1,"1,885.000",47.000,"7,241.000",3.000,2.000,2.000,4.000,1.000,0.000,0.000,0.000,0.000,32.770,-117.056,0.000,0.000,0.000,0.000,0.000,"1,390,627.420","1,353,501.821","989,964.884"
2,"1,316.000",65.000,"7,241.000",3.000,2.000,2.000,4.000,1.000,0.000,0.000,0.000,0.000,32.722,-117.049,0.000,0.000,0.000,0.000,0.000,"789,405.284","1,345,594.076","791,130.323"
3,"1,642.000",0.000,"2,066.000",2.000,3.000,2.000,2.000,1.000,0.000,1.000,0.000,1.000,34.120,-118.232,1.000,0.000,0.000,0.000,0.000,"1,759,646.137","1,351,429.218","1,293,281.828"
4,"1,680.000",0.000,"1,804.000",3.000,3.000,2.000,2.000,1.000,0.000,1.000,0.000,1.000,34.119,-118.233,1.000,0.000,0.000,0.000,0.000,"1,746,048.972","1,351,117.864","1,283,315.208"


## 5. Things to bring back to the team

In [10]:
print("LotSizeSquareFeet over 100 acres:", (df["LotSizeSquareFeet"] > 100 * da.SQFT_PER_ACRE).sum())
print("PropertyAge below 0:", (df["PropertyAge"] < 0).sum(), "| above 150:", (df["PropertyAge"] > 150).sum())

LotSizeSquareFeet over 100 acres: 279
PropertyAge below 0: 27 | above 150: 27


- **Lot size has no upper limit.** Some lots are huge. That's fine for trees, but it pulls a linear model around unless the log-transform experiment is used.
- **PropertyAge has no range rule.** A home built the year after its close year gets a negative age (pre-construction sales). These are kept as they are.
- **Leftover `ListingKey` duplicates.** The address + date rule misses repeated listings whose `CloseDate` changed between monthly exports. That's the count printed in section 1.
- **Target encoding target.** The doc doesn't say whether to encode against price or log price. Here the encoder uses the target the model is trained on.

## 6. Differences from `02_preprocessing.ipynb`
| | 02 (individual) | 02b (Dataset A) |
|---|---|---|
| Duplicates | one row per `ListingKey` | same address + close date |
| ClosePrice outliers | fixed \$50K to \$50M + sale-to-list ratio | training 0.5% / 99.5% percentiles |
| Bad values | wider ranges, more fields | agreed ranges only |
| Features | adds HOA fee, levels, main-level beds, basement, MLS area, high school district | agreed list only |
| Missing flags | many columns | Y/N fields only |
| Counties | one-hot | target encoded |